# Notebook to test and train


In [1]:
from pathlib import Path
import sys

project_root = Path().resolve().parent
sys.path.append(str(project_root))

1. Import data

In [2]:
from src.featureEngineering.feature_encoding import Encode
from src.dataExtraction.extract import split, import_data
from src.featureEngineering.outcome_labelling import outcome

# Import data

data_path = project_root / "data" / "raw" / "BPI_Challenge_2013_incidents" / "BPI_Challenge_2013_incidents.xes"
drop_columns = ["impact", "org:role"]
df = import_data(str(data_path), drop_columns=drop_columns)




/Users/so/Desktop/explainable-process-prediction/.venv/lib/python3.13/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
/Users/so/Desktop/explainable-process-prediction/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 7554/7554 [00:01<00:00, 6525.48it/s]


      org:group resource country organization country org:resource  \
0           V30           France                   fr     Frederic   
1           V30           France                   fr     Frederic   
2        V5 3rd           France                   fr     Frederic   
3        V5 3rd           France                   fr  Anne Claire   
4           V30           France                   fr  Anne Claire   
...         ...              ...                  ...          ...   
65528        C9           Brazil                   br      Lierson   
65529        C9           Brazil                   br      Lierson   
65530        C9           Brazil                   br      Lierson   
65531        C9           Brazil                   br      Lierson   
65532       N36              USA                   us         Matt   

      organization involved concept:name  product lifecycle:transition  \
0               Org line A2     Accepted  PROD582          In Progress   
1          

2. Define outcome

In [3]:
df = df.copy()

case_duration = (
    df.groupby("case:concept:name")["time:timestamp"]
    .agg(lambda x: (x.max() - x.min()).total_seconds())
)

df["case_duration_seconds"] = df["case:concept:name"].map(case_duration)

# Median split: positive outcome = shorter/equal median duration.

duration_threshold = df.groupby("case:concept:name")["case_duration_seconds"].first().median()
outcome_rules = [
    {
        "feature": "case_duration_seconds",
        "operator": "always_le",
        "value": duration_threshold,
    }
]

df = outcome(df, outcome_rules)


3. perform the split

In [4]:
train_df, val_df, test_df = split(df)

X_train, y_train, case_ids_train, feature_columns = Encode(train_df)

X_val, y_val, case_ids_val, _ = Encode(
    val_df,
    feature_columns=feature_columns
)

X_test, y_test, case_ids_test, _ = Encode(
    test_df,
    feature_columns=feature_columns
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

print(len(feature_columns))

(5287, 145)
(1133, 145)
(1134, 145)
145


In [5]:
print(y_train.value_counts())
print(y_val.value_counts())
print(y_test.value_counts())

outcome
False    3226
True     2061
Name: count, dtype: int64
outcome
True     845
False    288
Name: count, dtype: int64
outcome
True     871
False    263
Name: count, dtype: int64


At this point split works, features are stable, rule-based labelings works.
The next step would be to train and evaluate the models.

## Logistic regression

In [6]:
from src.modeling.models import train_logistic_regression

logreg = train_logistic_regression(X_train, y_train)
print(logreg)

LogisticRegression(class_weight='balanced', max_iter=20000, random_state=42)


/Users/so/Desktop/explainable-process-prediction/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 12841 iteration(s) (status=1):
STOP: TOTAL NO. OF F,G EVALUATIONS EXCEEDS LIMIT

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


The model fails to converge, maybe scaling and/or increasing number of iterations is necessary

In [7]:
from src.modeling.evaluation import evaluate_model

logreg_val_result = evaluate_model(
    logreg,
    X_val,
    y_val,
    "Logistic Regression",
)

logreg_val_result

{'accuracy': 0.9770520741394528,
 'f1': 0.9843937575030012,
 'precision': 0.9987819732034104,
 'recall': 0.9704142011834319,
 'roc_auc': 0.9983933267587114,
 'model': 'Logistic Regression'}

In [8]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg_scaled = make_pipeline(
    StandardScaler(with_mean=False),
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    ),
)

logreg_scaled.fit(X_train, y_train)

scaled_result = evaluate_model(
    logreg_scaled,
    X_val,
    y_val,
    "Logistic Regression scaled",
)

scaled_result

{'accuracy': 0.8340688437775816,
 'f1': 0.8754966887417218,
 'precision': 0.9939849624060151,
 'recall': 0.7822485207100591,
 'roc_auc': 0.9648504273504274,
 'model': 'Logistic Regression scaled'}

Logistic regression showed good performance but does not converge, likely due to unscaled heterogeneous features. A scaled variant was tested but performed worse on validation data.

## Random Forest

In [9]:
from src.modeling.model_selection import select_best_random_forest

best_rf, best_params, validation_table = select_best_random_forest(
    X_train,
    y_train,
    X_val,
    y_val,
    scoring="f1",
)

print("Best Random Forest parameters:")
print(best_params)

validation_table

Best Random Forest parameters:
{'n_estimators': 100, 'max_depth': None}


,accuracy,f1,precision,recall,roc_auc,model,params
0,1.0,1.0,1.0,1.0,1.0,"Random Forest {'n_estimators': 100, 'max_depth...","{'n_estimators': 100, 'max_depth': None}"
1,1.0,1.0,1.0,1.0,1.0,"Random Forest {'n_estimators': 100, 'max_depth...","{'n_estimators': 100, 'max_depth': 5}"
2,1.0,1.0,1.0,1.0,1.0,"Random Forest {'n_estimators': 100, 'max_depth...","{'n_estimators': 100, 'max_depth': 10}"
3,1.0,1.0,1.0,1.0,1.0,"Random Forest {'n_estimators': 200, 'max_depth...","{'n_estimators': 200, 'max_depth': None}"
4,1.0,1.0,1.0,1.0,1.0,"Random Forest {'n_estimators': 200, 'max_depth...","{'n_estimators': 200, 'max_depth': 5}"
5,1.0,1.0,1.0,1.0,1.0,"Random Forest {'n_estimators': 200, 'max_depth...","{'n_estimators': 200, 'max_depth': 10}"


The results of both models are too good which suggests feature leakage. Try an experiment - drop duration-based features

In [10]:
time_leakage_features = [
    col for col in feature_columns
    if any(
        keyword in col.lower()
        for keyword in [
            "elapsed_time",
            "relative_age",
            "time_since_last",
            "case_duration",
        ]
    )
]

print(time_leakage_features)
print(len(time_leakage_features))

['case_duration_seconds_sum', 'case_duration_seconds_mean', 'case_duration_seconds_min', 'case_duration_seconds_max', 'case_duration_seconds_std', 'case_duration_seconds_count', 'elapsed_time_sum', 'elapsed_time_mean', 'elapsed_time_min', 'elapsed_time_max', 'elapsed_time_std', 'elapsed_time_count', 'time_since_last_sum', 'time_since_last_mean', 'time_since_last_min', 'time_since_last_max', 'time_since_last_std', 'time_since_last_count', 'relative_age_sum', 'relative_age_mean', 'relative_age_min', 'relative_age_max', 'relative_age_std', 'relative_age_count']
24


In [11]:
keep_feature_indices = [
    i for i, col in enumerate(feature_columns)
    if col not in time_leakage_features
]

feature_columns_no_leakage = [
    feature_columns[i] for i in keep_feature_indices
]

X_train_no_leakage = X_train[:, keep_feature_indices]
X_val_no_leakage = X_val[:, keep_feature_indices]
X_test_no_leakage = X_test[:, keep_feature_indices]

print(X_train_no_leakage.shape)
print(X_val_no_leakage.shape)
print(X_test_no_leakage.shape)

(5287, 121)
(1133, 121)
(1134, 121)


In [12]:
best_rf, best_params, validation_table = select_best_random_forest(
    X_train_no_leakage,
    y_train,
    X_val_no_leakage,
    y_val,
    scoring="f1",
)

print(best_params)
validation_table

{'n_estimators': 200, 'max_depth': 5}


,accuracy,f1,precision,recall,roc_auc,model,params
0,0.707855,0.761355,0.974170,0.624852,0.847660,"Random Forest {'n_estimators': 100, 'max_depth...","{'n_estimators': 100, 'max_depth': None}"
1,0.729038,0.782424,0.975265,0.653254,0.918130,"Random Forest {'n_estimators': 100, 'max_depth...","{'n_estimators': 100, 'max_depth': 5}"
2,0.709620,0.765168,0.964029,0.634320,0.870607,"Random Forest {'n_estimators': 100, 'max_depth...","{'n_estimators': 100, 'max_depth': 10}"
3,0.717564,0.770445,0.978142,0.635503,0.854845,"Random Forest {'n_estimators': 200, 'max_depth...","{'n_estimators': 200, 'max_depth': None}"
4,0.735216,0.789621,0.969019,0.666272,0.924474,"Random Forest {'n_estimators': 200, 'max_depth...","{'n_estimators': 200, 'max_depth': 5}"
5,0.710503,0.766049,0.964093,0.635503,0.866465,"Random Forest {'n_estimators': 200, 'max_depth...","{'n_estimators': 200, 'max_depth': 10}"


At first, the Random Forest got perfect validation scores. This looked suspicious, because the outcome was based on case duration and some features also contained duration-related information. To avoid this leakage, features such as elapsed time, relative age, and time since last event were removed. After that, the validation results became more realistic. The best Random Forest configuration according to validation F1 was n_estimators=200 and max_depth=5.

Now retrain logreg model

In [14]:
from src.modeling.models import train_logistic_regression
from src.modeling.evaluation import evaluate_model

logreg_no_leakage = train_logistic_regression(
    X_train_no_leakage,
    y_train
)

logreg_no_leakage_result = evaluate_model(
    logreg_no_leakage,
    X_val_no_leakage,
    y_val,
    "Logistic Regression",
)

logreg_no_leakage_result

{'accuracy': 0.8428949691085613,
 'f1': 0.8927710843373494,
 'precision': 0.90920245398773,
 'recall': 0.8769230769230769,
 'roc_auc': 0.8975016436554898,
 'model': 'Logistic Regression'}

The logistic regression results after removing leakage are looking more realistic too.

## Evaluation and comparison

In [15]:
from src.modeling.evaluation import build_evaluation_table

best_rf_val_result = validation_table.loc[
    validation_table["params"].astype(str) == str(best_params)
].iloc[0].to_dict()

validation_comparison = build_evaluation_table([
    logreg_no_leakage_result,
    best_rf_val_result,
])

validation_comparison


,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.842895,0.909202,0.876923,0.892771,0.897502
1,"Random Forest {'n_estimators': 200, 'max_depth...",0.735216,0.969019,0.666272,0.789621,0.924474


Log reg has better accuracy, recall and f1

RF has better precision and ROC AUC

In [16]:
from src.modeling.evaluation import build_evaluation_table

best_rf_val_result = validation_table.loc[
    validation_table["params"].astype(str) == str(best_params)
].iloc[0].to_dict()

validation_comparison = build_evaluation_table([
    logreg_no_leakage_result,
    best_rf_val_result,
])

validation_comparison

,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.842895,0.909202,0.876923,0.892771,0.897502
1,"Random Forest {'n_estimators': 200, 'max_depth...",0.735216,0.969019,0.666272,0.789621,0.924474


In [18]:
test_results = [
    evaluate_model(
        logreg_no_leakage,
        X_test_no_leakage,
        y_test,
        "Logistic Regression",
    ),
    evaluate_model(
        best_rf,
        X_test_no_leakage,
        y_test,
        "Random Forest",
    ),
]

test_table = build_evaluation_table(test_results)
test_table

,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.804233,0.933244,0.802526,0.862963,0.893938
1,Random Forest,0.670194,0.962756,0.593571,0.734375,0.881516
